# Notebook 03 — 1D Linear Interpolation and Time-Series Imputation

**Session:** Interpolation and Image upsampling  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement 1D linear interpolation from two known anchor points $p_1 = (x_1, y_1)$ and
$p_2 = (x_2, y_2)$, distinguish finding the *first* missing index (`break`) from finding
the *last*, and impute arbitrary missing `None` runs in 1D time-series data from
`../data/time_series_missing.json`.

In [ ]:
import json
from pathlib import Path

with Path("../data/time_series_missing.json").open(encoding="utf-8") as fp:
    series_data = json.load(fp)

## 1. 1D Linear Interpolation Formula

In [ ]:
def linear_interpolation(p1: tuple[float, float], p2: tuple[float, float], x: float) -> float:
    """Linearly interpolate y at *x* given endpoints p1=(x1, y1) and p2=(x2, y2)."""
    x1, y1 = p1
    x2, y2 = p2
    if x1 == x2:
        return float(y1)
    return ((x2 - x) / (x2 - x1)) * y1 + ((x - x1) / (x2 - x1)) * y2


examples = [
    ((1, 1), (3, 3), 2),
    ((1, 1), (3, 3), 1.1),
    ((1, 1), (6, 9), 3),
    ((1, 2.2), (3, 2.7), 2),
]
for p1, p2, x_val in examples:
    y_val = linear_interpolation(p1, p2, x_val)
    print(f"p1={p1}, p2={p2}, x={x_val} -> y={y_val:.4f}")

## 2. Finding the First Missing Value (`break` Invariant)

In [ ]:
def find_last_none(data: list[float | None]) -> int:
    """Return the LAST index containing None (illustrating missing break)."""
    result = -1
    for i, val in enumerate(data):
        if val is None:
            result = i
    return result


def find_first_none(data: list[float | None]) -> int:
    """Return the FIRST index containing None, or -1 if none exist."""
    for i, val in enumerate(data):
        if val is None:
            return i
    return -1


for sample in [
    [2.0, None, 2.2, 2.7, 6.2],
    [2.0, 2.2, None, None, 2.7, 6.2],
    [2.0, 2.2, 3.2, 2.7, 6.2, 5.9],
]:
    print(sample, "-> last_none:", find_last_none(sample), "| first_none:", find_first_none(sample))

## 3. Complete Time-Series Imputation Algorithm

In [ ]:
def find_next_valid(data: list[float | None], start_idx: int) -> int:
    """Return the first index > *start_idx* with a non-None value, or len(data)."""
    for i in range(start_idx + 1, len(data)):
        if data[i] is not None:
            return i
    return len(data)


def impute_linear_1d(data: list[float | None]) -> list[float]:
    """Fill None entries in-place on a copy using linear interpolation and edge clamping."""
    filled = list(data)
    while True:
        index = find_first_none(filled)
        if index == -1:
            break

        begin = index - 1
        end = find_next_valid(filled, index)

        if begin < 0 and end >= len(filled):
            raise ValueError("Series contains only None values; cannot interpolate.")
        if begin < 0:
            y = float(filled[end])  # type: ignore[arg-type]
        elif end >= len(filled):
            y = float(filled[begin])  # type: ignore[arg-type]
        else:
            p1 = (float(begin), float(filled[begin]))  # type: ignore[arg-type]
            p2 = (float(end), float(filled[end]))  # type: ignore[arg-type]
            y = linear_interpolation(p1=p1, p2=p2, x=float(index))

        filled[index] = y
    return [float(v) for v in filled]


imputed = impute_linear_1d(series_data["boundary_missing_series"])
assert all(
    abs(a - b) < 1e-9 for a, b in zip(imputed, series_data["expected_linear_imputed"], strict=True)
)
print("Imputed boundary_missing_series:", [round(v, 3) for v in imputed])

imputed_trailing = impute_linear_1d(series_data["trailing_missing_series"])
print("Imputed trailing_missing_series:", [round(v, 3) for v in imputed_trailing])